# Week 1 Concepts — Computer Vision & Time Series

Runnable companion notebook for Days 1-4. Each section uses a different
worked example than its matching daily note, demonstrating the same
underlying technique.

## Day 1: CNNs vs. Vision Transformers

Below: classifying a batch of trail-camera wildlife photos with a small
pretrained ViT, building a results table, and flagging low-confidence
predictions for manual review.

In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import torch
import pandas as pd

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")

camera_trap_photos = ["trailcam_0091.jpg", "trailcam_0092.jpg", "trailcam_0093.jpg"]

rows = []
for path in camera_trap_photos:
    image = Image.open(path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = logits.softmax(dim=-1)[0]
    top_prob, top_idx = probs.max(dim=-1)
    rows.append({
        "file": path,
        "predicted_label": model.config.id2label[top_idx.item()],
        "confidence": round(top_prob.item(), 3),
        "needs_review": top_prob.item() < 0.6,
    })

results = pd.DataFrame(rows)
results

## Day 2: Diffusion Models and Text-to-Image Generation

Below: batch-generating sticker concepts for a chat app from a few
prompts, comparing `num_inference_steps` and `guidance_scale` settings
in one comparison table.

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import pandas as pd

pipe = StableDiffusionPipeline.from_pretrained(
    "segmind/small-sd",  # distilled, lightweight checkpoint
    torch_dtype=torch.float32,
)

sticker_prompts = [
    "a cartoon sticker of a sleepy cat wearing headphones, flat vector style",
    "a cartoon sticker of a rocket ship with a smiling face, flat vector style",
]

rows = []
for prompt in sticker_prompts:
    for steps, guidance in [(15, 4.0), (30, 7.5), (30, 12.0)]:
        image = pipe(prompt, num_inference_steps=steps, guidance_scale=guidance).images[0]
        fname = f"sticker_{sticker_prompts.index(prompt)}_{steps}_{guidance}.png"
        image.save(fname)
        rows.append({"prompt": prompt, "steps": steps, "guidance_scale": guidance, "file": fname})

comparison = pd.DataFrame(rows)
comparison

## Day 3: Time Series Fundamentals

Below: a synthetic hourly electricity-demand series with daily
seasonality, decomposed with `seasonal_decompose`, then anomaly-checked
on raw vs. deseasonalized data to see the seasonal-peak false-positive
problem directly.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

rng = pd.date_range("2024-01-01", periods=24 * 60, freq="h")
trend = np.linspace(300, 340, len(rng))                         # slow demand growth
daily_cycle = 60 * np.sin(2 * np.pi * rng.hour / 24)             # evening-peak daily pattern
noise = np.random.normal(0, 8, len(rng))
electricity_demand = pd.Series(trend + daily_cycle + noise, index=rng, name="megawatts")

decomposition = seasonal_decompose(electricity_demand, model="additive", period=24)
decomposition.trend.tail(), decomposition.seasonal.head(24)

In [ ]:
# Anomaly check on RAW data: every evening peak looks "anomalous"
roll_mean_raw = electricity_demand.rolling(48).mean()
roll_std_raw = electricity_demand.rolling(48).std()
raw_anomalies = (electricity_demand - roll_mean_raw).abs() > 3 * roll_std_raw
print("Raw-data anomalies flagged:", raw_anomalies.sum())

# Anomaly check on the DESEASONALIZED residual instead
resid = decomposition.resid.dropna()
roll_mean_resid = resid.rolling(48).mean()
roll_std_resid = resid.rolling(48).std()
resid_anomalies = (resid - roll_mean_resid).abs() > 3 * roll_std_resid
print("Residual-based anomalies flagged:", resid_anomalies.sum())

## Day 4: Forecasting Methods and Evaluation

Below: forecasting monthly newsletter subscriber growth (trend +
yearly seasonality) with Holt-Winters, holding out the last 12 months,
simulating a prediction interval, and scoring against a naive
moving-average baseline.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

rng = pd.date_range("2019-01-01", periods=72, freq="MS")
trend = np.linspace(500, 4000, len(rng))
yearly_cycle = 200 * np.sin(2 * np.pi * rng.month / 12)
noise = np.random.normal(0, 60, len(rng))
subscribers = pd.Series(trend + yearly_cycle + noise, index=rng, name="new_subscribers")

train, test = subscribers[:-12], subscribers[-12:]

hw_model = ExponentialSmoothing(
    train, trend="add", seasonal="add", seasonal_periods=12
).fit()
hw_forecast = hw_model.forecast(len(test))

simulations = hw_model.simulate(len(test), repetitions=200, error="add")
lower, upper = simulations.quantile(0.05, axis=1), simulations.quantile(0.95, axis=1)

baseline_forecast = pd.Series(train.rolling(3).mean().iloc[-1], index=test.index)

def mae(y, yhat): return np.mean(np.abs(y - yhat))
def rmse(y, yhat): return np.sqrt(np.mean((y - yhat) ** 2))

print("Holt-Winters  MAE:", round(mae(test, hw_forecast), 1), "RMSE:", round(rmse(test, hw_forecast), 1))
print("Moving-avg    MAE:", round(mae(test, baseline_forecast), 1), "RMSE:", round(rmse(test, baseline_forecast), 1))